# Multiple Dataset Diagnostics

Note: These diagnostics and fixed-parameter test runs were executed on a different machine with an 8× NVIDIA RTX A6000 GPU cluster; therefore, small numerical/runtime variation from the main reported results may be observed. On the first run, TabPFN may download the required v2.5 checkpoint from Hugging Face; the checkpoint is cached afterward, and setting an HF_TOKEN can avoid unauthenticated-rate-limit warnings.

In [2]:
# ============================================================
# Single-cell notebook script: GOTabPFN package diagnostics
#
# Drops non-numeric feature columns automatically
# ============================================================

import os, sys, gc, random, warnings
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import importlib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import package and diagnostics module through current package setup
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

diag = gotabpfn.load_dataset_diagnostics_module()

print("[OK] Imported gotabpfn package.")
print("[OK] Loaded gotabpfn_dataset_diagnostics.py through package.")

# ------------------------------------------------------------
# Patch loader: drop target, then keep numeric feature columns only
# ------------------------------------------------------------
def load_numeric_csv_drop_non_numeric(
    csv_path,
    target_col=None,
    standardize=True,
):
    df = pd.read_csv(csv_path)

    target_col = diag._none_if_empty(target_col)

    if target_col is not None:
        if target_col not in df.columns:
            raise ValueError(
                f"Target column '{target_col}' not found in {csv_path}.\n"
                f"Available columns: {list(df.columns)}"
            )
        df = df.drop(columns=[target_col])

    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c])
    ]

    dropped_cols = [
        c for c in df.columns
        if c not in numeric_cols
    ]

    if dropped_cols:
        print(f"[INFO] {csv_path}: dropped non-numeric columns: {dropped_cols}")

    if len(numeric_cols) < 2:
        raise ValueError(
            f"{csv_path}: expected at least two numeric feature columns after dropping "
            f"target/non-numeric columns. Found {len(numeric_cols)}."
        )

    X = df[numeric_cols].to_numpy(dtype=np.float32)

    X = np.nan_to_num(
        X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)

    if standardize:
        X = StandardScaler().fit_transform(X).astype(np.float32)

    return X

# Replace original loader inside diagnostics module
diag.load_numeric_csv = load_numeric_csv_drop_non_numeric

# ------------------------------------------------------------
# Dataset list
# ------------------------------------------------------------
datasets = [
    {
        "csv_path": "cellcycle.csv",
        "target_col": "phase",
        "dataset_name": "Cell Cycle",
    },
    {
        "csv_path": "BASEHOCK.csv",
        "target_col": "label",
        "dataset_name": "BASEHOCK",
    },
    {
        "csv_path": "RELATHE.csv",
        "target_col": "label",
        "dataset_name": "RELATHE",
    },
    {
        "csv_path": "PCMAC.csv",
        "target_col": "label",
        "dataset_name": "PCMAC",
    },
    {
        "csv_path": "orlraws10P.csv",
        "target_col": "label",
        "dataset_name": "orlraws10P",
    },
]

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------
missing = [d["csv_path"] for d in datasets if not os.path.exists(d["csv_path"])]

if missing:
    raise FileNotFoundError(
        "Missing dataset file(s):\n" + "\n".join([f"  - {f}" for f in missing])
    )

print("[OK] All dataset files found.")

# ------------------------------------------------------------
# Run diagnostics
# ------------------------------------------------------------
df_metrics = diag.analyze_many_csvs(
    datasets=datasets,
    out_csv="cross_domain_ordering_metrics.csv",
    standardize=True,
    verbose=True,
)

# ------------------------------------------------------------
# Select final columns
# ------------------------------------------------------------
show_cols = [
    "FOE_rank",
    "dataset",
    "category",
    "n",
    "m",
    "rho",
    "IDF_final",
    "FOE",
    "P_success",
    "Delta_AdjCoh",
    "Delta_HitRate",
    "Delta_Cut",
    "LES",
    "AUC",
]

df_show = df_metrics[show_cols].copy()

# Save selected table
df_show.to_csv("cross_domain_ordering_metrics_selected_columns.csv", index=False)

print("\n[SAVED]")
print("  - cross_domain_ordering_metrics.csv")
print("  - cross_domain_ordering_metrics_selected_columns.csv")

# Pretty display
display(
    df_show.style.format({
        "rho": "{:.4g}",
        "IDF_final": "{:.4g}",
        "FOE": "{:.4g}",
        "P_success": "{:.4g}",
        "Delta_AdjCoh": "{:.4g}",
        "Delta_HitRate": "{:.4g}",
        "Delta_Cut": "{:.4g}",
        "LES": "{:.4g}",
        "AUC": "{:.4g}",
    })
)

[OK] Imported gotabpfn package.
[OK] Loaded gotabpfn_dataset_diagnostics.py through package.
[OK] All dataset files found.
[INFO] cellcycle.csv: dropped non-numeric columns: ['cell']
[INFO] Dataset=Cell Cycle | category=MixedRegime | n=1067 | m=42728 | rho=40.045
[INFO] Dataset=BASEHOCK | category=MixedRegime | n=1993 | m=4862 | rho=2.43954
[INFO] Dataset=RELATHE | category=MixedRegime | n=1427 | m=4322 | rho=3.02873
[INFO] Dataset=PCMAC | category=MixedRegime | n=1943 | m=3289 | rho=1.69274
[INFO] Dataset=orlraws10P | category=HDLSS | n=100 | m=10304 | rho=103.04
[SAVED] cross_domain_ordering_metrics.csv

[SAVED]
  - cross_domain_ordering_metrics.csv
  - cross_domain_ordering_metrics_selected_columns.csv


,FOE_rank,dataset,category,n,m,rho,IDF_final,FOE,P_success,Delta_AdjCoh,Delta_HitRate,Delta_Cut,LES,AUC
0,1,orlraws10P,HDLSS,100,10304,103,0.009317,1.152e+04,0.9907,0.003487,0.001641,-0.02229,-0.1697,0.005118
1,2,Cell Cycle,MixedRegime,1067,42728,40.04,0.02439,1681,0.9756,0.00867,0.0001758,0.000777,0.6586,0.004773
2,3,RELATHE,MixedRegime,1427,4322,3.029,0.2915,11.77,0.7085,-0.001728,-0.0005078,-0.00457,-0.6643,0.1484
3,4,BASEHOCK,MixedRegime,1993,4862,2.44,0.3616,7.649,0.6384,0.0002262,0.000127,0.005039,0.03449,0.1852
4,5,PCMAC,MixedRegime,1943,3289,1.693,0.5132,3.796,0.4868,-0.000303,0.00252,-0.01112,0.1409,0.2612


# Single dataset diagnostic

In [3]:
# ============================================================
# Single-cell notebook script: GOTabPFN package diagnostics for ONE dataset
# Drops non-numeric feature columns automatically
# ============================================================

import os, sys, gc, random, warnings
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import importlib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# User input: change these three lines only
# ------------------------------------------------------------
DATA_FILE = "cellcycle.csv"
TARGET_COL = "phase"
DATASET_NAME = "Cell Cycle"

OUT_CSV = f"{DATASET_NAME.replace(' ', '_').replace('/', '_')}_single_ordering_metrics.csv"

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import package and diagnostics module through current package setup
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

diag = gotabpfn.load_dataset_diagnostics_module()

print("[OK] Imported gotabpfn package.")
print("[OK] Loaded gotabpfn_dataset_diagnostics.py through package.")

# ------------------------------------------------------------
# Patch loader: drop target, then keep numeric feature columns only
# ------------------------------------------------------------
def load_numeric_csv_drop_non_numeric(
    csv_path,
    target_col=None,
    standardize=True,
):
    df = pd.read_csv(csv_path)

    target_col = diag._none_if_empty(target_col)

    if target_col is not None:
        if target_col not in df.columns:
            raise ValueError(
                f"Target column '{target_col}' not found in {csv_path}.\n"
                f"Available columns: {list(df.columns)}"
            )
        df = df.drop(columns=[target_col])

    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c])
    ]

    dropped_cols = [
        c for c in df.columns
        if c not in numeric_cols
    ]

    if dropped_cols:
        print(f"[INFO] {csv_path}: dropped non-numeric columns: {dropped_cols}")

    if len(numeric_cols) < 2:
        raise ValueError(
            f"{csv_path}: expected at least two numeric feature columns after dropping "
            f"target/non-numeric columns. Found {len(numeric_cols)}."
        )

    X = df[numeric_cols].to_numpy(dtype=np.float32)

    X = np.nan_to_num(
        X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)

    if standardize:
        X = StandardScaler().fit_transform(X).astype(np.float32)

    return X

# Replace original loader inside diagnostics module
diag.load_numeric_csv = load_numeric_csv_drop_non_numeric

# ------------------------------------------------------------
# Check file
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Missing dataset file: {DATA_FILE}")

print(f"[OK] Dataset file found: {DATA_FILE}")

# ------------------------------------------------------------
# Run diagnostics for one dataset only
# LES for single dataset is handled inside your diagnostics file:
# finite mean of Delta_AdjCoh, Delta_HitRate, Delta_Cut.
# ------------------------------------------------------------
df_metrics = diag.analyze_csv_ordering_metrics(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    dataset_name=DATASET_NAME,
    out_csv=OUT_CSV,
    standardize=True,
    verbose=True,
)

# ------------------------------------------------------------
# Select final columns
# ------------------------------------------------------------
show_cols = [
    "FOE_rank",
    "dataset",
    "category",
    "n",
    "m",
    "rho",
    "IDF_final",
    "FOE",
    "P_success",
    "Delta_AdjCoh",
    "Delta_HitRate",
    "Delta_Cut",
    "LES",
    "AUC",
]

df_show = df_metrics[show_cols].copy()

# Save selected table
selected_csv = OUT_CSV.replace(".csv", "_selected_columns.csv")
df_show.to_csv(selected_csv, index=False)

print("\n[SAVED]")
print(f"  - {OUT_CSV}")
print(f"  - {selected_csv}")

# ------------------------------------------------------------
# Pretty display
# ------------------------------------------------------------
display(
    df_show.style.format({
        "rho": "{:.4g}",
        "IDF_final": "{:.4g}",
        "FOE": "{:.4g}",
        "P_success": "{:.4g}",
        "Delta_AdjCoh": "{:.4g}",
        "Delta_HitRate": "{:.4g}",
        "Delta_Cut": "{:.4g}",
        "LES": "{:.4g}",
        "AUC": "{:.4g}",
    })
)

[OK] Imported gotabpfn package.
[OK] Loaded gotabpfn_dataset_diagnostics.py through package.
[OK] Dataset file found: cellcycle.csv
[INFO] cellcycle.csv: dropped non-numeric columns: ['cell']
[INFO] Dataset=Cell Cycle | category=MixedRegime | n=1067 | m=42728 | rho=40.045
[SAVED] Cell_Cycle_single_ordering_metrics.csv

[SAVED]
  - Cell_Cycle_single_ordering_metrics.csv
  - Cell_Cycle_single_ordering_metrics_selected_columns.csv


,FOE_rank,dataset,category,n,m,rho,IDF_final,FOE,P_success,Delta_AdjCoh,Delta_HitRate,Delta_Cut,LES,AUC
0,1,Cell Cycle,MixedRegime,1067,42728,40.04,0.02439,1681,0.9756,0.00867,0.0001758,0.000777,0.003208,0.004773


# GO-LR Ordering as Metaheuristics

In [4]:
# ============================================================
# Single-cell notebook test for GO-LR through current gotabpfn package
# Tests ordering runtime, TSP path cost, MinLA cost, and reordered features
#
# This code is tested on RTX A6000 GPU, which may differ from the
# original device used for paper experiments. A faster runtime may be expected.
# ============================================================

import os
import sys
import warnings
import importlib
import pandas as pd

warnings.filterwarnings(
    "ignore",
    message=".*pynvml package is deprecated.*",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*cumsum_cuda_kernel does not have a deterministic implementation.*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*Deterministic behavior was enabled.*CuBLAS.*",
    category=UserWarning,
)

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import package
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

print("[OK] Imported gotabpfn package.")

# ------------------------------------------------------------
# Config: same GO-LR settings from Colon ablation
# ------------------------------------------------------------
SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
DATASET_NAME = "Colon"

BEST_GO = {
    "metric": "euclidean",
    "num_clusters": 10,
    "refine_passes": 3,
    "direction_select": True,
}

OUT_PREFIX = "colon_golr_test"

# ------------------------------------------------------------
# Check files / exports
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE}")

if getattr(gotabpfn, "run_golr_csv", None) is None:
    raise ImportError("gotabpfn.run_golr_csv is not available. Check __init__.py and GO-LR.py.")

print(f"[OK] Found dataset: {DATA_FILE}")
print("[OK] Found gotabpfn.run_golr_csv")

# ------------------------------------------------------------
# Run GO-LR
# ------------------------------------------------------------
result = gotabpfn.run_golr_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    dataset_name=DATASET_NAME,
    metric=BEST_GO["metric"],
    num_clusters=BEST_GO["num_clusters"],
    refine=True,
    direction_select=BEST_GO["direction_select"],
    refine_passes=BEST_GO["refine_passes"],
    bins=32,
    seed=SEED,
    standardize=True,
    drop_non_numeric=True,
    use_cpu_kmeans=True,   # safer for notebook testing
    save_outputs=True,
    out_prefix=OUT_PREFIX,
)

# ------------------------------------------------------------
# Display metrics
# ------------------------------------------------------------
metrics = result["metrics"]
metrics_df = pd.DataFrame([metrics])

print("\n[GO-LR metrics]")
display(metrics_df)

# ------------------------------------------------------------
# Display learned ordering preview
# ------------------------------------------------------------
ordering_df = result["ordering_df"]
print("\n[Ordering preview]")
display(ordering_df.head(20))

# ------------------------------------------------------------
# Display reordered feature table preview
# ------------------------------------------------------------
reordered_df = result["reordered_df"]
print("\n[Reordered feature table preview]")
display(reordered_df.head())

# ------------------------------------------------------------
# Access important values directly
# ------------------------------------------------------------
Pi_star = result["ordering"]
runtime_sec = result["runtime_sec"]
tsp_cost = result["tsp_path_cost"]
minla_cost = result["minla_cost"]

print("\n[Direct values]")
print(f"Number of ordered features: {len(Pi_star)}")
print(f"Runtime seconds: {runtime_sec:.6f}")
print(f"TSP path cost: {tsp_cost:.6f}")
print(f"MinLA cost: {minla_cost:.6f}")

print("\n[SAVED]")
print(f"  - {OUT_PREFIX}_reordered.csv")
print(f"  - {OUT_PREFIX}_ordering.csv")
print(f"  - {OUT_PREFIX}_metrics.json")

[OK] Imported gotabpfn package.
[OK] Found dataset: coloncancer_encoded.csv
[OK] Found gotabpfn.run_golr_csv
[GO-LR] Dataset: Colon
[GO-LR] X shape: (62, 2000)
[GO-LR] metric=euclidean, num_clusters=10, refine=True, direction_select=True, refine_passes=3

================ GO-LR SUMMARY ================
dataset: Colon
n: 62
m: 2000
metric: euclidean
num_clusters: 10
refine: True
direction_select: True
refine_passes: 3
runtime_sec: 8.266112202996737
tsp_path_cost: 21921.666637063026
minla_cost: 14751002624.0

[SAVED]
  - colon_golr_test_reordered.csv
  - colon_golr_test_ordering.csv
  - colon_golr_test_metrics.json

[GO-LR metrics]


,dataset,n,m,metric,num_clusters,refine,direction_select,refine_passes,runtime_sec,tsp_path_cost,minla_cost
0,Colon,62,2000,euclidean,10,True,True,3,8.266112,21921.666637,1.475100e+10



[Ordering preview]


,rank,feature_index,feature_name
0,1,308,309
1,2,675,676
2,3,512,513
3,4,844,845
4,5,1667,1668
5,6,548,549
6,7,573,574
7,8,552,553
8,9,1593,1594
9,10,1186,1187



[Reordered feature table preview]


,309,676,513,845,1668,549,574,553,1594,1187,...,1141,1757,963,71,1796,1765,1359,1219,1823,759
0,0.072582,0.656352,0.340945,-0.122847,0.947328,0.639425,0.075959,1.362230,-0.914930,-0.723345,...,-0.018881,0.621647,0.544623,1.604201,1.015544,0.455010,-0.844512,2.075658,0.197368,-0.933047
1,-0.364162,0.564583,-0.871813,1.139789,0.681475,0.468621,-0.489650,0.821872,-1.264599,-1.518032,...,0.916659,-0.056893,0.361593,0.094625,0.687518,0.812415,0.272188,0.586499,0.199505,0.099005
2,1.927437,1.224424,-2.546066,-0.907695,2.012698,0.312187,1.524534,1.557138,-2.433363,-1.347743,...,1.177879,1.297950,-0.378140,-1.617902,1.857300,-1.553953,0.544251,0.092630,-0.286850,-1.053894
3,1.663439,2.480152,-1.338367,-0.052560,1.302084,1.066415,-0.572707,1.781293,-2.567197,-1.101499,...,0.267576,0.167595,0.373477,-2.164740,0.506348,0.651415,0.193691,-0.363324,0.599027,-0.167270
4,0.123675,-0.274034,1.494230,-0.820308,1.194684,-0.355270,0.210089,0.472459,-0.415526,0.127764,...,-1.087638,-2.470091,1.330171,0.462949,-0.308354,-0.597506,-0.558495,0.526132,-2.712402,0.609672



[Direct values]
Number of ordered features: 2000
Runtime seconds: 8.266112
TSP path cost: 21921.666637
MinLA cost: 14751002624.000000

[SAVED]
  - colon_golr_test_reordered.csv
  - colon_golr_test_ordering.csv
  - colon_golr_test_metrics.json


# Compression check

In [5]:
# ============================================================
# Test all 4 NSC variants on Colon dataset
# Tests:
#   NSC-pSP
#   NSC-SP
#   NSC-P
#   NSC
# ============================================================

import os
import sys
import gc
import warnings
import importlib

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

warnings.filterwarnings(
    "ignore",
    message=".*pynvml package is deprecated.*",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*cumsum_cuda_kernel does not have a deterministic implementation.*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*Deterministic behavior was enabled.*CuBLAS.*",
    category=UserWarning,
)

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import package
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

print("[OK] Imported gotabpfn package.")

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
DATASET_NAME = "Colon"

ORDERING_CSV = "colon_golr_ordering.csv"
SEED = 42

NSC_COMMON = {
    "segmentation": "equal_mass",
    "m_rule": "idf",
    "gamma": 1.7570143129240916,
    "beta": 0.2244046472232107,
    "M_min": 64,
    "M_max": 384,
    "l_min": 16,
    "standardize_input": True,
    "drop_non_numeric": True,
    "mode": "flatten",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_outputs": True,
}

DESC_COMMON = {
    "descriptor": "basic",
    "pooling": "learn_free",
}

TAU = 0.99

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def maybe_make_golr_ordering():
    if os.path.exists(ORDERING_CSV):
        print(f"[OK] Found ordering file: {ORDERING_CSV}")
        return ORDERING_CSV

    if getattr(gotabpfn, "run_golr_csv", None) is None:
        print("[WARN] gotabpfn.run_golr_csv is unavailable.")
        print("[WARN] Falling back to identity ordering for all NSC variants.")
        return None

    print(f"[INFO] {ORDERING_CSV} not found. Running GO-LR through gotabpfn package...")

    gotabpfn.run_golr_csv(
        csv_path=DATA_FILE,
        target_col=TARGET_COL,
        dataset_name=DATASET_NAME,
        metric="euclidean",
        num_clusters=10,
        refine=True,
        direction_select=True,
        refine_passes=3,
        bins=32,
        seed=SEED,
        standardize=True,
        drop_non_numeric=True,
        use_cpu_kmeans=True,
        save_outputs=True,
        out_prefix="colon_golr",
    )

    if os.path.exists(ORDERING_CSV):
        print(f"[OK] Created ordering file: {ORDERING_CSV}")
        return ORDERING_CSV

    print("[WARN] GO-LR ran, but ordering file was not found.")
    print("[WARN] Falling back to identity ordering for all NSC variants.")
    return None


# ------------------------------------------------------------
# Check dataset and package exports
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Missing dataset file: {DATA_FILE}")

required_exports = [
    "run_nsc_psp_csv",
    "run_nsc_sp_csv",
    "run_nsc_p_csv",
    "run_nsc_csv",
]

missing_exports = [x for x in required_exports if getattr(gotabpfn, x, None) is None]
if missing_exports:
    raise ImportError(
        "Missing required gotabpfn exports:\n"
        + "\n".join([f"  - {x}" for x in missing_exports])
    )

print(f"[OK] Found dataset: {DATA_FILE}")
print("[OK] Found all NSC wrapper exports.")

ordering_csv = maybe_make_golr_ordering()

# ------------------------------------------------------------
# Run all four variants
# ------------------------------------------------------------
results = {}

print("\n" + "=" * 80)
print("Running NSC-pSP")
print("=" * 80)

results["NSC-pSP"] = gotabpfn.run_nsc_psp_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    tau=TAU,
    out_prefix="colon_nsc_psp_test",
    **NSC_COMMON,
)

M_ref = int(results["NSC-pSP"]["metrics"]["M_selected"])
d_hat_ref = results["NSC-pSP"]["metrics"].get("d_hat_pca", None)

print(f"\n[REFERENCE from NSC-pSP] M_ref={M_ref}, d_hat_ref={d_hat_ref}")

cleanup()

print("\n" + "=" * 80)
print("Running NSC-SP")
print("=" * 80)

results["NSC-SP"] = gotabpfn.run_nsc_sp_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    M=M_ref,
    out_prefix="colon_nsc_sp_test",
    **NSC_COMMON,
)

cleanup()

print("\n" + "=" * 80)
print("Running NSC-P")
print("=" * 80)

results["NSC-P"] = gotabpfn.run_nsc_p_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    tau=TAU,
    out_prefix="colon_nsc_p_test",
    **NSC_COMMON,
    **DESC_COMMON,
)

cleanup()

print("\n" + "=" * 80)
print("Running NSC")
print("=" * 80)

results["NSC"] = gotabpfn.run_nsc_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    M=M_ref,
    estimate_d_hat=False,
    out_prefix="colon_nsc_test",
    **NSC_COMMON,
    **DESC_COMMON,
)

cleanup()

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------
summary_rows = []

for name, res in results.items():
    m = res["metrics"].copy()
    m["variant"] = name
    summary_rows.append(m)

summary_df = pd.DataFrame(summary_rows)

preferred_cols = [
    "variant",
    "dataset",
    "n",
    "m_original",
    "m_compressed",
    "compression_ratio",
    "M_selected",
    "idf",
    "d_hat_pca",
    "segmentation",
    "m_rule",
    "descriptor",
    "pooling",
    "runtime_sec",
    "ordering_source",
]

available_cols = [c for c in preferred_cols if c in summary_df.columns]
remaining_cols = [c for c in summary_df.columns if c not in available_cols]
summary_df = summary_df[available_cols + remaining_cols]

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

display(
    summary_df.style.format({
        "compression_ratio": "{:.4g}",
        "idf": "{:.4g}",
        "d_hat_pca": "{:.4g}",
        "runtime_sec": "{:.4g}",
    }, na_rep="NA")
)

summary_df.to_csv("colon_all_nsc_variants_summary.csv", index=False)

# ------------------------------------------------------------
# Preview compressed outputs
# ------------------------------------------------------------
for name, res in results.items():
    print(f"\n[{name}] compressed_df preview:")
    display(res["compressed_df"].head())

print("\n[SAVED SUMMARY]")
print("  - colon_all_nsc_variants_summary.csv")

print("\n[SAVED COMPRESSED FILES]")
print("  - colon_nsc_psp_test_compressed.csv")
print("  - colon_nsc_sp_test_compressed.csv")
print("  - colon_nsc_p_test_compressed.csv")
print("  - colon_nsc_test_compressed.csv")

print("\n[SAVED SEGMENTS/METRICS]")
print("  - *_segments.csv")
print("  - *_metrics.json")

[OK] Imported gotabpfn package.
[OK] Found dataset: coloncancer_encoded.csv
[OK] Found all NSC wrapper exports.
[OK] Found ordering file: colon_golr_ordering.csv

Running NSC-pSP
[NSC-pSP] Dataset: Colon
[NSC-pSP] X shape: (62, 2000)
[NSC-pSP] ordering: colon_golr_ordering.csv
[NSC-pSP] segmentation=equal_mass, m_rule=idf, tau=0.99

================ NSC-pSP SUMMARY ================
dataset: Colon
n: 62
m_original: 2000
m_compressed: 70
compression_ratio: 28.571428571428573
ordering_source: colon_golr_ordering.csv
segmentation: equal_mass
m_rule: idf
tau: 0.99
gamma: 1.7570143129240916
beta: 0.2244046472232107
M_min: 64
M_max: 384
l_min: 16
M_selected: 70
idf: 0.0285
d_hat_pca: 57.0
runtime_sec: 0.14072773800580762

[SAVED]
  - colon_nsc_psp_test_compressed.csv
  - colon_nsc_psp_test_segments.csv
  - colon_nsc_psp_test_metrics.json

[REFERENCE from NSC-pSP] M_ref=70, d_hat_ref=57.0

Running NSC-SP
[NSC-SP] Dataset: Colon
[NSC-SP] X shape: (62, 2000)
[NSC-SP] ordering: colon_golr_orderin

,variant,dataset,n,m_original,m_compressed,compression_ratio,M_selected,idf,d_hat_pca,segmentation,m_rule,descriptor,pooling,runtime_sec,ordering_source,tau,gamma,beta,M_min,M_max,l_min,d_hat,M_given,d_hat_given,d_hat_given_or_estimated,d_hat_source
0,NSC-pSP,Colon,62,2000,70,28.57,70,0.0285,57,equal_mass,idf,NA,NA,0.1407,colon_golr_ordering.csv,0.990000,1.757014,0.224405,64,384,16,NA,NA,NA,NA,NA
1,NSC-SP,Colon,62,2000,70,28.57,70,NA,NA,equal_mass,idf,NA,NA,0.03772,colon_golr_ordering.csv,NA,1.757014,0.224405,64,384,16,NA,70.000000,NA,NA,NA
2,NSC-P,Colon,62,2000,70,28.57,70,0.0285,57,equal_mass,idf,basic,learn_free,0.02076,colon_golr_ordering.csv,0.990000,1.757014,0.224405,64,384,16,NA,NA,NA,NA,NA
3,NSC,Colon,62,2000,70,28.57,70,NA,NA,equal_mass,idf,basic,learn_free,0.01634,colon_golr_ordering.csv,NA,1.757014,0.224405,64,384,16,NA,70.000000,NA,NA,not_used



[NSC-pSP] compressed_df preview:


,z_001,z_002,z_003,z_004,z_005,z_006,z_007,z_008,z_009,z_010,...,z_061,z_062,z_063,z_064,z_065,z_066,z_067,z_068,z_069,z_070
0,4.129642,4.820275,3.531628,3.569583,3.583393,3.141495,-2.146232,2.515643,4.172479,3.485234,...,-2.124541,-0.326527,0.281295,3.439358,-0.892282,2.910153,1.666409,2.349433,0.111373,1.005579
1,4.063647,3.269246,3.173676,2.318845,2.890347,2.300746,-4.769880,3.132932,2.883978,3.586185,...,0.256524,1.472289,1.832616,0.112400,-2.676220,1.833951,1.908866,-1.585966,-1.388330,0.739065
2,7.751984,8.229130,6.404116,6.647443,6.750544,6.602628,-6.303511,4.434593,7.486387,7.610234,...,-2.164552,-1.472708,2.365332,-1.254610,3.694949,1.651543,0.737908,2.472412,2.876192,0.550870
3,7.701643,6.248197,7.685204,6.844953,7.618235,6.613824,-4.834740,5.950317,5.253567,6.773824,...,-2.876985,-1.549101,1.411175,2.474636,-1.114025,-1.178000,0.405933,-0.823298,-0.185018,0.587684
4,-1.008051,2.308615,-1.807310,-1.966315,-1.583946,-0.129079,2.310877,-1.228599,0.063494,-1.099781,...,0.598392,-1.262395,-1.427069,-1.540671,1.448907,-3.751735,-5.037189,-2.743999,2.612844,-4.080699



[NSC-SP] compressed_df preview:


...


[NSC-P] compressed_df preview:


,z_001,z_002,z_003,z_004,z_005,z_006,z_007,z_008,z_009,z_010,...,z_061,z_062,z_063,z_064,z_065,z_066,z_067,z_068,z_069,z_070
0,0.364786,0.451454,0.341403,0.171555,0.196201,0.482519,0.081112,0.282334,0.067373,0.329145,...,-0.085820,0.086903,0.131532,0.314572,0.132640,0.404863,0.221156,0.381663,-0.019290,0.276753
1,0.076595,0.325854,0.217170,0.076638,0.010685,0.200901,-0.112916,0.229506,0.120799,0.130243,...,0.176955,0.159798,0.351697,0.139125,0.302062,0.209334,0.488893,0.400314,0.014266,0.101316
2,0.512462,0.779755,0.184414,0.452203,0.594412,0.404169,-0.096767,0.394211,0.449145,0.403336,...,0.216174,0.396160,0.211032,0.110716,0.412651,0.401043,0.182776,0.447224,0.753382,0.252028
3,0.577770,0.644089,0.682162,0.464953,0.503429,0.300215,0.046266,0.506337,0.415201,0.450070,...,0.289831,-0.101136,0.182429,0.320891,0.379595,0.248763,0.113725,0.357125,0.322893,0.105904
4,0.222534,0.418726,0.153653,0.313958,0.504704,0.276233,0.263940,0.329368,0.333258,0.338439,...,-0.014434,0.023524,-0.064100,0.045058,0.152097,-0.321843,-0.041978,-0.048733,0.083108,-0.015057



[NSC] compressed_df preview:


,z_001,z_002,z_003,z_004,z_005,z_006,z_007,z_008,z_009,z_010,...,z_061,z_062,z_063,z_064,z_065,z_066,z_067,z_068,z_069,z_070
0,0.364786,0.451454,0.341403,0.171555,0.196201,0.482519,0.081112,0.282334,0.067373,0.329145,...,-0.085820,0.086903,0.131532,0.314572,0.132640,0.404863,0.221156,0.381663,-0.019290,0.276753
1,0.076595,0.325854,0.217170,0.076638,0.010685,0.200901,-0.112916,0.229506,0.120799,0.130243,...,0.176955,0.159798,0.351697,0.139125,0.302062,0.209334,0.488893,0.400314,0.014266,0.101316
2,0.512462,0.779755,0.184414,0.452203,0.594412,0.404169,-0.096767,0.394211,0.449145,0.403336,...,0.216174,0.396160,0.211032,0.110716,0.412651,0.401043,0.182776,0.447224,0.753382,0.252028
3,0.577770,0.644089,0.682162,0.464953,0.503429,0.300215,0.046266,0.506337,0.415201,0.450070,...,0.289831,-0.101136,0.182429,0.320891,0.379595,0.248763,0.113725,0.357125,0.322893,0.105904
4,0.222534,0.418726,0.153653,0.313958,0.504704,0.276233,0.263940,0.329368,0.333258,0.338439,...,-0.014434,0.023524,-0.064100,0.045058,0.152097,-0.321843,-0.041978,-0.048733,0.083108,-0.015057



[SAVED SUMMARY]
  - colon_all_nsc_variants_summary.csv

[SAVED COMPRESSED FILES]
  - colon_nsc_psp_test_compressed.csv
  - colon_nsc_sp_test_compressed.csv
  - colon_nsc_p_test_compressed.csv
  - colon_nsc_test_compressed.csv

[SAVED SEGMENTS/METRICS]
  - *_segments.csv
  - *_metrics.json


# Multiclass Classification || orlaws10P dataset || 10 class

In [6]:
# =======================
# Fixed-parameter 5x5 CV w/o Optuna tuning
# Current gotabpfn package setup
# =======================

import os
import sys
import gc
import time
import random
import warnings
import importlib

GPU_ID = 2  # run on cuda:2 if available

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import current package setup
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

GraphFeatureOrdering = gotabpfn.GraphFeatureOrdering
PIDFSegPCA = gotabpfn.pidf_segpca
TabPFN25Head = gotabpfn.TabPFN25Head
TabPFN25Config = gotabpfn.TabPFN25Config

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
SEED = 42
DATA_FILE = "orlraws10P.csv"
TARGET_COL = "label"

FIXED_PARAMS = {
    "go_metric": "cosine",
    "go_num_clusters": 5,
    "go_refine_passes": 1,
    "go_direction_select": False,
    "go_feat_subsample": 3000,
    "nsc_segmentation": "uniform",
    "nsc_m_rule": "default",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.049512863264476,
    "nsc_beta": 0.3887505167779042,
    "nsc_Mmin": 32,
    "nsc_Mmax": 384,
    "nsc_lmin": 12,
    "assume_standardized": False,
    "tabpfn_seed": 42,
}

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE_STR = f"cuda:{GPU_ID}"
else:
    DEVICE_STR = "cpu"

NSC_DEVICE = DEVICE_STR
TABPFN_DEVICE = DEVICE_STR

# ------------------------------------------------------------
# Utils
# ------------------------------------------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


def cleanup_cuda():
    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def compute_deltas_adjacent_corr_chunked(
    X_tr: np.ndarray,
    Pi_star: list[int],
    chunk: int = 2048,
    eps: float = 1e-12,
):
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    _, d = X.shape
    deltas = torch.empty(d - 1, dtype=torch.float32)

    mu = X.mean(dim=0)
    Xc = X - mu
    std = Xc.std(dim=0, unbiased=False).clamp_min(eps)

    for a in range(0, d - 1, chunk):
        b = min(d - 1, a + chunk)

        i_idx = perm[a:b]
        j_idx = perm[a + 1:b + 1]

        Zi = (Xc[:, i_idx] / std[i_idx]).to(torch.float32)
        Zj = (Xc[:, j_idx] / std[j_idx]).to(torch.float32)

        corr = (Zi * Zj).mean(dim=0)
        deltas[a:b] = (1.0 - corr.abs()).cpu()

    return deltas


def safe_multiclass_macro_ovr_auc(
    y_true: np.ndarray,
    proba: np.ndarray,
    num_classes: int,
):
    try:
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(
            roc_auc_score(
                y_bin,
                proba,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        return np.nan


# ============================================
# Load + preprocessing
# ============================================
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in {DATA_FILE}")

# Target
y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
target_le = LabelEncoder()
y_all = target_le.fit_transform(y_raw).astype(np.int64)

class_map = {cls: int(i) for i, cls in enumerate(target_le.classes_)}
NUM_CLASSES = len(target_le.classes_)

# Features
X_df = df.drop(columns=[TARGET_COL]).copy()

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
removed_cat_cols = [c for c in X_df.columns if c not in num_cols]

if len(num_cols) == 0:
    raise ValueError("No numeric feature columns found after removing target and non-numeric columns.")

X_num = X_df[num_cols].apply(pd.to_numeric, errors="coerce")
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num = X_num.apply(pd.to_numeric, errors="coerce").fillna(0.0)

scaler = StandardScaler()
X_all = scaler.fit_transform(X_num.values).astype(np.float32, copy=False)
X_all = np.asarray(X_all, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] Raw shape={df.shape}")
print(f"[DATA] Processed X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print(f"[DATA] Numeric cols kept={len(num_cols)} | Removed non-numeric cols={len(removed_cat_cols)}")

if len(removed_cat_cols) > 0:
    print(f"[DATA] Removed columns: {removed_cat_cols}")

print(
    "[GPU ] cuda_available=", torch.cuda.is_available(),
    "| visible_gpus=", torch.cuda.device_count(),
    "| using_device=", DEVICE_STR,
)

if NUM_CLASSES != 10:
    print(f"[WARN] Expected 10 classes, but found {NUM_CLASSES} classes.")

rkf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

m_full = X_all.shape[1]


# ============================================
# Build ordering once
# ============================================
def build_ordering(params: dict):
    go_metric = params["go_metric"]
    go_k = int(params["go_num_clusters"])
    go_refine_passes = int(params["go_refine_passes"])
    go_direction = bool(params["go_direction_select"])
    go_feat_subsample = int(params["go_feat_subsample"])

    rng = np.random.default_rng(SEED + 999)

    if go_feat_subsample < m_full:
        feat_idx = rng.choice(m_full, size=go_feat_subsample, replace=False)
        feat_idx.sort()
    else:
        feat_idx = np.arange(m_full)

    X_go = X_all[:, feat_idx]

    go = GraphFeatureOrdering(
        num_clusters=go_k,
        metric=go_metric,
        refine=True,
        direction_select=go_direction,
        refine_passes=go_refine_passes,
    )

    try:
        Pi_sub, _, _, _ = go.fit(
            X_go,
            seed=SEED,
            deterministic=True,
            use_cpu_kmeans=False,
        )
    except Exception:
        cleanup_cuda()
        Pi_sub, _, _, _ = go.fit(
            X_go,
            seed=SEED,
            deterministic=True,
            use_cpu_kmeans=True,
        )

    ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

    if go_feat_subsample < m_full:
        remaining = np.setdiff1d(
            np.arange(m_full),
            feat_idx,
            assume_unique=False,
        )
        Pi_star = ordered_subset + remaining.tolist()
    else:
        Pi_star = ordered_subset

    return list(map(int, Pi_star))


# ============================================
# Evaluate fixed params
# ============================================
def evaluate_with_params(params: dict, record_fold_metrics: bool = True):
    seed_everything(SEED)
    cleanup_cuda()

    Pi_star = build_ordering(params)

    tabpfn_seed = int(params["tabpfn_seed"])

    head_cfg = TabPFN25Config(
        task_type="multiclass",
        num_classes=int(NUM_CLASSES),
        device=TABPFN_DEVICE,
        random_state=tabpfn_seed,
    )

    accs = []
    f1s = []
    aucs = []
    fold_rows = []

    t0 = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        nsc = PIDFSegPCA(
            segmentation=params["nsc_segmentation"],
            l_min=int(params["nsc_lmin"]),
            m_rule=params["nsc_m_rule"],
            gamma=float(params["nsc_gamma"]),
            beta=float(params["nsc_beta"]),
            tau=float(params["nsc_tau"]),
            M_min=int(params["nsc_Mmin"]),
            M_max=int(params["nsc_Mmax"]),
            assume_standardized=bool(params["assume_standardized"]),
            device=NSC_DEVICE,
        )

        deltas = None

        if params["nsc_segmentation"] != "uniform":
            deltas = compute_deltas_adjacent_corr_chunked(
                X_tr,
                Pi_star,
                chunk=2048,
            )

        X_tr_t = torch.from_numpy(X_tr)

        nsc.configure(
            Pi_star=Pi_star,
            X_train=X_tr_t,
            tau=float(params["nsc_tau"]),
            deltas=deltas,
        )

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)

        acc = float(accuracy_score(y_va, pred))
        f1m = float(f1_score(y_va, pred, average="macro"))
        aucm = safe_multiclass_macro_ovr_auc(y_va, P, NUM_CLASSES)

        accs.append(acc)
        f1s.append(f1m)
        aucs.append(aucm)

        fold_rows.append({
            "fold": fold_id,
            "accuracy": acc,
            "macro_f1": f1m,
            "macro_ovr_auc": aucm,
        })

        if record_fold_metrics:
            print(
                f"[Fold {fold_id:02d}] "
                f"ACC={acc:.6f} | "
                f"Macro-F1={f1m:.6f} | "
                f"Macro-OvR-AUC={aucm:.6f}"
            )

        cleanup_cuda()

    elapsed_sec = time.perf_counter() - t0

    return {
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs, ddof=1)),
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s, ddof=1)),
        "auc_mean": float(np.nanmean(aucs)),
        "auc_std": float(np.nanstd(aucs, ddof=1)),
        "elapsed_sec": float(elapsed_sec),
        "fold_accs": accs,
        "fold_f1s": f1s,
        "fold_aucs": aucs,
        "fold_df": pd.DataFrame(fold_rows),
    }


print("\n================ FIXED PARAMETERS ================")
for k, v in FIXED_PARAMS.items():
    print(f"{k}: {v}")

print("\n================ FINAL 5x5 CV WITH FIXED PARAMS ================")

final_results = evaluate_with_params(
    FIXED_PARAMS,
    record_fold_metrics=True,
)

print("\n================ FINAL SUMMARY ================")
print(f"Device used: {DEVICE_STR}")
print(f"Elapsed time for fixed-param 5x5 CV: {final_results['elapsed_sec']:.2f} seconds")
print(f"Accuracy      : {final_results['acc_mean']:.6f} ± {final_results['acc_std']:.6f}")
print(f"Macro-F1      : {final_results['f1_mean']:.6f} ± {final_results['f1_std']:.6f}")
print(f"Macro ROC-AUC : {final_results['auc_mean']:.6f} ± {final_results['auc_std']:.6f}")

final_results["fold_df"].to_csv("orlraws10P_fixed_5x5_foldwise.csv", index=False)

summary_df = pd.DataFrame([{
    "dataset": DATA_FILE,
    "device": DEVICE_STR,
    "acc_mean": final_results["acc_mean"],
    "acc_std": final_results["acc_std"],
    "f1_mean": final_results["f1_mean"],
    "f1_std": final_results["f1_std"],
    "auc_mean": final_results["auc_mean"],
    "auc_std": final_results["auc_std"],
    "elapsed_sec": final_results["elapsed_sec"],
}])

summary_df.to_csv("orlraws10P_fixed_5x5_summary.csv", index=False)

print("\n[SAVED]")
print("  - orlraws10P_fixed_5x5_foldwise.csv")
print("  - orlraws10P_fixed_5x5_summary.csv")

[DATA] Raw shape=(100, 10305)
[DATA] Processed X=(100, 10304) | C=10 | map={'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
[DATA] Numeric cols kept=10304 | Removed non-numeric cols=0
[GPU ] cuda_available= True | visible_gpus= 8 | using_device= cuda:2

================ FIXED PARAMETERS ================
go_metric: cosine
go_num_clusters: 5
go_refine_passes: 1
go_direction_select: False
go_feat_subsample: 3000
nsc_segmentation: uniform
nsc_m_rule: default
nsc_tau: 0.99
nsc_gamma: 2.049512863264476
nsc_beta: 0.3887505167779042
nsc_Mmin: 32
nsc_Mmax: 384
nsc_lmin: 12
assume_standardized: False
tabpfn_seed: 42

================ FINAL 5x5 CV WITH FIXED PARAMS ================
[Fold 01] ACC=1.000000 | Macro-F1=1.000000 | Macro-OvR-AUC=1.000000
[Fold 02] ACC=1.000000 | Macro-F1=1.000000 | Macro-OvR-AUC=1.000000
[Fold 03] ACC=1.000000 | Macro-F1=1.000000 | Macro-OvR-AUC=1.000000
[Fold 04] ACC=1.000000 | Macro-F1=1.000000 | Macro-OvR-AUC=1.000000
[Fold 05] ACC=1.

# Binary Classification || Colon dataset

In [7]:
# =======================
# Fixed-parameter 5x5 CV
# Current gotabpfn package setup
# Dataset: Colon
# =======================

import os
import sys
import gc
import time
import random
import warnings
import importlib

GPU_ID = 2  # change if needed

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import current package setup
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

GraphFeatureOrdering = gotabpfn.GraphFeatureOrdering
PIDFSegPCA = gotabpfn.pidf_segpca
TabPFN25Head = gotabpfn.TabPFN25Head
TabPFN25Config = gotabpfn.TabPFN25Config

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"

FIXED_PARAMS = {
    "go_metric": "euclidean",
    "go_num_clusters": 10,
    "go_refine_passes": 3,
    "go_direction_select": True,
    "go_feat_subsample": None,

    "nsc_segmentation": "equal_mass",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 1.7570143129240916,
    "nsc_beta": 0.2244046472232107,
    "nsc_Mmin": 64,
    "nsc_Mmax": 384,
    "nsc_lmin": 16,
    "assume_standardized": False,

    "tabpfn_seed": 42,
}

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE_STR = f"cuda:{GPU_ID}"
else:
    DEVICE_STR = "cpu"

NSC_DEVICE = DEVICE_STR
TABPFN_DEVICE = DEVICE_STR

# ------------------------------------------------------------
# Utils
# ------------------------------------------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


def cleanup_cuda():
    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def compute_deltas_adjacent_corr_chunked(
    X_tr: np.ndarray,
    Pi_star: list[int],
    chunk: int = 2048,
    eps: float = 1e-12,
):
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    _, d = X.shape
    deltas = torch.empty(d - 1, dtype=torch.float32)

    mu = X.mean(dim=0)
    Xc = X - mu
    std = Xc.std(dim=0, unbiased=False).clamp_min(eps)

    for a in range(0, d - 1, chunk):
        b = min(d - 1, a + chunk)

        i_idx = perm[a:b]
        j_idx = perm[a + 1:b + 1]

        Zi = (Xc[:, i_idx] / std[i_idx]).to(torch.float32)
        Zj = (Xc[:, j_idx] / std[j_idx]).to(torch.float32)

        corr = (Zi * Zj).mean(dim=0)
        deltas[a:b] = (1.0 - corr.abs()).cpu()

    return deltas


def safe_binary_auc(y_true: np.ndarray, proba: np.ndarray):
    try:
        if proba.ndim == 2 and proba.shape[1] > 1:
            score = proba[:, 1]
        else:
            score = proba.reshape(-1)
        return float(roc_auc_score(y_true, score))
    except Exception:
        return np.nan


def safe_multiclass_macro_ovr_auc(
    y_true: np.ndarray,
    proba: np.ndarray,
    num_classes: int,
):
    try:
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(
            roc_auc_score(
                y_bin,
                proba,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        return np.nan


def safe_auc(y_true: np.ndarray, proba: np.ndarray, num_classes: int):
    if num_classes == 2:
        return safe_binary_auc(y_true, proba)
    return safe_multiclass_macro_ovr_auc(y_true, proba, num_classes)


# ============================================
# Load + preprocessing
# ============================================
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in {DATA_FILE}")

# Target
y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
target_le = LabelEncoder()
y_all = target_le.fit_transform(y_raw).astype(np.int64)

class_map = {cls: int(i) for i, cls in enumerate(target_le.classes_)}
NUM_CLASSES = len(target_le.classes_)

# Features
X_df = df.drop(columns=[TARGET_COL]).copy()

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
removed_cat_cols = [c for c in X_df.columns if c not in num_cols]

if len(num_cols) == 0:
    raise ValueError("No numeric feature columns found after removing target and non-numeric columns.")

X_num = X_df[num_cols].apply(pd.to_numeric, errors="coerce")
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num = X_num.apply(pd.to_numeric, errors="coerce").fillna(0.0)

scaler = StandardScaler()
X_all = scaler.fit_transform(X_num.values).astype(np.float32, copy=False)
X_all = np.asarray(X_all, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] Raw shape={df.shape}")
print(f"[DATA] Processed X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print(f"[DATA] Numeric cols kept={len(num_cols)} | Removed non-numeric cols={len(removed_cat_cols)}")

if len(removed_cat_cols) > 0:
    print(f"[DATA] Removed columns: {removed_cat_cols}")

print(
    "[GPU ] cuda_available=", torch.cuda.is_available(),
    "| visible_gpus=", torch.cuda.device_count(),
    "| using_device=", DEVICE_STR,
)

rkf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

m_full = X_all.shape[1]


# ============================================
# Build ordering once
# ============================================
def build_ordering(params: dict):
    go_metric = params["go_metric"]
    go_k = int(params["go_num_clusters"])
    go_refine_passes = int(params["go_refine_passes"])
    go_direction = bool(params["go_direction_select"])
    go_feat_subsample = params.get("go_feat_subsample", None)

    if go_feat_subsample is not None and int(go_feat_subsample) < m_full:
        go_feat_subsample = int(go_feat_subsample)

        rng = np.random.default_rng(SEED + 999)
        feat_idx = rng.choice(m_full, size=go_feat_subsample, replace=False)
        feat_idx.sort()
    else:
        feat_idx = np.arange(m_full)

    X_go = X_all[:, feat_idx]

    go = GraphFeatureOrdering(
        num_clusters=go_k,
        metric=go_metric,
        refine=True,
        direction_select=go_direction,
        refine_passes=go_refine_passes,
    )

    try:
        Pi_sub, _, _, _ = go.fit(
            X_go,
            seed=SEED,
            deterministic=True,
            use_cpu_kmeans=False,
        )
    except Exception:
        cleanup_cuda()
        Pi_sub, _, _, _ = go.fit(
            X_go,
            seed=SEED,
            deterministic=True,
            use_cpu_kmeans=True,
        )

    ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

    if len(feat_idx) < m_full:
        remaining = np.setdiff1d(
            np.arange(m_full),
            feat_idx,
            assume_unique=False,
        )
        Pi_star = ordered_subset + remaining.tolist()
    else:
        Pi_star = ordered_subset

    return list(map(int, Pi_star))


# ============================================
# Evaluate fixed params
# ============================================
def evaluate_with_params(params: dict, record_fold_metrics: bool = True):
    seed_everything(SEED)
    cleanup_cuda()

    t_order = time.perf_counter()
    Pi_star = build_ordering(params)
    ordering_sec = time.perf_counter() - t_order

    tabpfn_seed = int(params["tabpfn_seed"])

    head_cfg = TabPFN25Config(
        task_type="binary" if NUM_CLASSES == 2 else "multiclass",
        num_classes=int(NUM_CLASSES),
        device=TABPFN_DEVICE,
        random_state=tabpfn_seed,
    )

    accs = []
    f1s = []
    aucs = []
    fold_rows = []

    t0 = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        nsc = PIDFSegPCA(
            segmentation=params["nsc_segmentation"],
            l_min=int(params["nsc_lmin"]),
            m_rule=params["nsc_m_rule"],
            gamma=float(params["nsc_gamma"]),
            beta=float(params["nsc_beta"]),
            tau=float(params["nsc_tau"]),
            M_min=int(params["nsc_Mmin"]),
            M_max=int(params["nsc_Mmax"]),
            assume_standardized=bool(params["assume_standardized"]),
            device=NSC_DEVICE,
        )

        deltas = None

        if params["nsc_segmentation"] != "uniform":
            deltas = compute_deltas_adjacent_corr_chunked(
                X_tr,
                Pi_star,
                chunk=2048,
            )

        X_tr_t = torch.from_numpy(X_tr)

        nsc.configure(
            Pi_star=Pi_star,
            X_train=X_tr_t,
            tau=float(params["nsc_tau"]),
            deltas=deltas,
        )

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)

        acc = float(accuracy_score(y_va, pred))
        f1m = float(f1_score(y_va, pred, average="macro"))
        aucm = safe_auc(y_va, P, NUM_CLASSES)

        accs.append(acc)
        f1s.append(f1m)
        aucs.append(aucm)

        fold_rows.append({
            "fold": fold_id,
            "accuracy": acc,
            "macro_f1": f1m,
            "auc": aucm,
            "m_compressed": int(Z_tr.shape[1]),
            "M_selected": int(nsc.M),
            "d_hat_pca": None if nsc.d_hat_pca_ is None else float(nsc.d_hat_pca_),
            "idf": None if nsc.idf_ is None else float(nsc.idf_),
        })

        if record_fold_metrics:
            print(
                f"[Fold {fold_id:02d}] "
                f"ACC={acc:.6f} | "
                f"Macro-F1={f1m:.6f} | "
                f"AUC={aucm:.6f}"
            )

        cleanup_cuda()

    elapsed_sec = time.perf_counter() - t0

    return {
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs, ddof=1)),
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s, ddof=1)),
        "auc_mean": float(np.nanmean(aucs)),
        "auc_std": float(np.nanstd(aucs, ddof=1)),
        "ordering_sec": float(ordering_sec),
        "elapsed_sec": float(elapsed_sec),
        "fold_accs": accs,
        "fold_f1s": f1s,
        "fold_aucs": aucs,
        "fold_df": pd.DataFrame(fold_rows),
    }


print("\n================ FIXED PARAMETERS ================")
for k, v in FIXED_PARAMS.items():
    print(f"{k}: {v}")

print("\n================ FINAL 5x5 CV WITH FIXED PARAMS ================")

final_results = evaluate_with_params(
    FIXED_PARAMS,
    record_fold_metrics=True,
)

print("\n================ FINAL SUMMARY ================")
print(f"Device used: {DEVICE_STR}")
print(f"Ordering time: {final_results['ordering_sec']:.2f} seconds")
print(f"Elapsed time for fixed-param 5x5 CV: {final_results['elapsed_sec']:.2f} seconds")
print(f"Accuracy : {final_results['acc_mean']:.6f} ± {final_results['acc_std']:.6f}")
print(f"Macro-F1 : {final_results['f1_mean']:.6f} ± {final_results['f1_std']:.6f}")
print(f"AUC      : {final_results['auc_mean']:.6f} ± {final_results['auc_std']:.6f}")

final_results["fold_df"].to_csv("colon_fixed_5x5_foldwise.csv", index=False)

summary_df = pd.DataFrame([{
    "dataset": DATA_FILE,
    "device": DEVICE_STR,
    "acc_mean": final_results["acc_mean"],
    "acc_std": final_results["acc_std"],
    "f1_mean": final_results["f1_mean"],
    "f1_std": final_results["f1_std"],
    "auc_mean": final_results["auc_mean"],
    "auc_std": final_results["auc_std"],
    "ordering_sec": final_results["ordering_sec"],
    "elapsed_sec": final_results["elapsed_sec"],
}])

summary_df.to_csv("colon_fixed_5x5_summary.csv", index=False)

print("\n[SAVED]")
print("  - colon_fixed_5x5_foldwise.csv")
print("  - colon_fixed_5x5_summary.csv")

[DATA] Raw shape=(62, 2001)
[DATA] Processed X=(62, 2000) | C=2 | map={'0': 0, '1': 1}
[DATA] Numeric cols kept=2000 | Removed non-numeric cols=0
[GPU ] cuda_available= True | visible_gpus= 8 | using_device= cuda:2

================ FIXED PARAMETERS ================
go_metric: euclidean
go_num_clusters: 10
go_refine_passes: 3
go_direction_select: True
go_feat_subsample: None
nsc_segmentation: equal_mass
nsc_m_rule: idf
nsc_tau: 0.99
nsc_gamma: 1.7570143129240916
nsc_beta: 0.2244046472232107
nsc_Mmin: 64
nsc_Mmax: 384
nsc_lmin: 16
assume_standardized: False
tabpfn_seed: 42

================ FINAL 5x5 CV WITH FIXED PARAMS ================
[Fold 01] ACC=1.000000 | Macro-F1=1.000000 | AUC=1.000000
[Fold 02] ACC=0.846154 | Macro-F1=0.837500 | AUC=0.900000
[Fold 03] ACC=0.916667 | Macro-F1=0.911111 | AUC=0.968750
[Fold 04] ACC=0.833333 | Macro-F1=0.812500 | AUC=0.812500
[Fold 05] ACC=1.000000 | Macro-F1=1.000000 | AUC=1.000000
[Fold 06] ACC=1.000000 | Macro-F1=1.000000 | AUC=1.000000
[Fold 0

# Regression || DrivFace dataset

In [2]:
# =======================
# Fixed-parameter 5x5 CV
# Current gotabpfn package setup
# Dataset: DrivFace Regression
# Metric: R^2
# Uses TabPFN25Head with task_type="regression"
# =======================

import os
import sys
import gc
import time
import random
import warnings
import importlib

GPU_ID = 2  # change if needed

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import current package setup
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

GraphFeatureOrdering = gotabpfn.GraphFeatureOrdering
PIDFSegPCA = gotabpfn.pidf_segpca
TabPFN25Head = gotabpfn.TabPFN25Head
TabPFN25Config = gotabpfn.TabPFN25Config

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
SEED = 42
DATA_FILE = "drivface.csv"
TARGET_COL = "angle"

FIXED_PARAMS = {
    "go_metric": "manhattan",
    "go_num_clusters": 5,
    "go_refine_passes": 1,
    "go_direction_select": False,
    "go_feat_subsample": 2000,

    "nsc_segmentation": "largest_jump",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.654390393837633,
    "nsc_beta": 0.043192175152615336,
    "nsc_Mmin": 16,
    "nsc_Mmax": 256,
    "nsc_lmin": 12,
    "assume_standardized": True,

    "tabpfn_seed": 3,
}

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE_STR = f"cuda:{GPU_ID}"
else:
    DEVICE_STR = "cpu"

NSC_DEVICE = DEVICE_STR
TABPFN_DEVICE = DEVICE_STR

# ------------------------------------------------------------
# Utils
# ------------------------------------------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


def cleanup_cuda():
    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def compute_deltas_adjacent_corr_chunked(
    X_tr: np.ndarray,
    Pi_star: list[int],
    chunk: int = 2048,
    eps: float = 1e-12,
):
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    _, d = X.shape
    deltas = torch.empty(d - 1, dtype=torch.float32)

    mu = X.mean(dim=0)
    Xc = X - mu
    std = Xc.std(dim=0, unbiased=False).clamp_min(eps)

    for a in range(0, d - 1, chunk):
        b = min(d - 1, a + chunk)

        i_idx = perm[a:b]
        j_idx = perm[a + 1:b + 1]

        Zi = (Xc[:, i_idx] / std[i_idx]).to(torch.float32)
        Zj = (Xc[:, j_idx] / std[j_idx]).to(torch.float32)

        corr = (Zi * Zj).mean(dim=0)
        deltas[a:b] = (1.0 - corr.abs()).cpu()

    return deltas


# ============================================
# Load + preprocessing
# ============================================
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in {DATA_FILE}")

# Target: regression target
y_raw = pd.to_numeric(df[TARGET_COL], errors="coerce")
if y_raw.isna().any():
    y_raw = y_raw.fillna(y_raw.median())

y_all = y_raw.to_numpy(dtype=np.float32)

# Features
X_df = df.drop(columns=[TARGET_COL]).copy()

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
removed_cat_cols = [c for c in X_df.columns if c not in num_cols]

if len(num_cols) == 0:
    raise ValueError("No numeric feature columns found after removing target and non-numeric columns.")

X_num = X_df[num_cols].apply(pd.to_numeric, errors="coerce")
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num = X_num.apply(pd.to_numeric, errors="coerce").fillna(0.0)

scaler = StandardScaler()
X_all = scaler.fit_transform(X_num.values).astype(np.float32, copy=False)
X_all = np.asarray(X_all, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.float32)

print(f"[DATA] Raw shape={df.shape}")
print(f"[DATA] Processed X={X_all.shape} | y={y_all.shape}")
print(f"[DATA] Numeric cols kept={len(num_cols)} | Removed non-numeric cols={len(removed_cat_cols)}")

if len(removed_cat_cols) > 0:
    print(f"[DATA] Removed columns: {removed_cat_cols}")

print(
    "[GPU ] cuda_available=", torch.cuda.is_available(),
    "| visible_gpus=", torch.cuda.device_count(),
    "| using_device=", DEVICE_STR,
)

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

m_full = X_all.shape[1]


# ============================================
# Build ordering once
# ============================================
def build_ordering(params: dict):
    go_metric = params["go_metric"]
    go_k = int(params["go_num_clusters"])
    go_refine_passes = int(params["go_refine_passes"])
    go_direction = bool(params["go_direction_select"])
    go_feat_subsample = int(params["go_feat_subsample"])

    rng = np.random.default_rng(SEED + 999)

    if go_feat_subsample < m_full:
        feat_idx = rng.choice(m_full, size=go_feat_subsample, replace=False)
        feat_idx.sort()
    else:
        feat_idx = np.arange(m_full)

    X_go = X_all[:, feat_idx]

    go = GraphFeatureOrdering(
        num_clusters=go_k,
        metric=go_metric,
        refine=True,
        direction_select=go_direction,
        refine_passes=go_refine_passes,
    )

    try:
        Pi_sub, _, _, _ = go.fit(
            X_go,
            seed=SEED,
            deterministic=True,
            use_cpu_kmeans=False,
        )
    except Exception:
        cleanup_cuda()
        Pi_sub, _, _, _ = go.fit(
            X_go,
            seed=SEED,
            deterministic=True,
            use_cpu_kmeans=True,
        )

    ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

    if go_feat_subsample < m_full:
        remaining = np.setdiff1d(
            np.arange(m_full),
            feat_idx,
            assume_unique=False,
        )
        Pi_star = ordered_subset + remaining.tolist()
    else:
        Pi_star = ordered_subset

    return list(map(int, Pi_star))


# ============================================
# Evaluate fixed params
# ============================================
def evaluate_with_params(params: dict, record_fold_metrics: bool = True):
    seed_everything(SEED)
    cleanup_cuda()

    t_order = time.perf_counter()
    Pi_star = build_ordering(params)
    ordering_sec = time.perf_counter() - t_order

    r2s = []
    rmses = []
    maes = []
    fold_rows = []

    head_cfg = TabPFN25Config(
        task_type="regression",
        num_classes=1,
        device=TABPFN_DEVICE,
        random_state=int(params["tabpfn_seed"]),
    )

    t0 = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all), start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        nsc = PIDFSegPCA(
            segmentation=params["nsc_segmentation"],
            l_min=int(params["nsc_lmin"]),
            m_rule=params["nsc_m_rule"],
            gamma=float(params["nsc_gamma"]),
            beta=float(params["nsc_beta"]),
            tau=float(params["nsc_tau"]),
            M_min=int(params["nsc_Mmin"]),
            M_max=int(params["nsc_Mmax"]),
            assume_standardized=bool(params["assume_standardized"]),
            device=NSC_DEVICE,
        )

        deltas = None

        if params["nsc_segmentation"] != "uniform":
            deltas = compute_deltas_adjacent_corr_chunked(
                X_tr,
                Pi_star,
                chunk=2048,
            )

        X_tr_t = torch.from_numpy(X_tr)

        nsc.configure(
            Pi_star=Pi_star,
            X_train=X_tr_t,
            tau=float(params["nsc_tau"]),
            deltas=deltas,
        )

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        pred = np.asarray(head.predict(Z_va), dtype=np.float32).reshape(-1)

        r2 = float(r2_score(y_va, pred))
        rmse = float(np.sqrt(mean_squared_error(y_va, pred)))
        mae = float(mean_absolute_error(y_va, pred))

        r2s.append(r2)
        rmses.append(rmse)
        maes.append(mae)

        fold_rows.append({
            "fold": fold_id,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "m_compressed": int(Z_tr.shape[1]),
            "M_selected": int(nsc.M),
            "d_hat_pca": None if nsc.d_hat_pca_ is None else float(nsc.d_hat_pca_),
            "idf": None if nsc.idf_ is None else float(nsc.idf_),
        })

        if record_fold_metrics:
            print(
                f"[Fold {fold_id:02d}] "
                f"R2={r2:.6f} | "
                f"RMSE={rmse:.6f} | "
                f"MAE={mae:.6f}"
            )

        cleanup_cuda()

    elapsed_sec = time.perf_counter() - t0

    return {
        "r2_mean": float(np.mean(r2s)),
        "r2_std": float(np.std(r2s, ddof=1)),
        "rmse_mean": float(np.mean(rmses)),
        "rmse_std": float(np.std(rmses, ddof=1)),
        "mae_mean": float(np.mean(maes)),
        "mae_std": float(np.std(maes, ddof=1)),
        "ordering_sec": float(ordering_sec),
        "elapsed_sec": float(elapsed_sec),
        "fold_r2s": r2s,
        "fold_rmses": rmses,
        "fold_maes": maes,
        "fold_df": pd.DataFrame(fold_rows),
    }


print("\n================ FIXED PARAMETERS ================")
for k, v in FIXED_PARAMS.items():
    print(f"{k}: {v}")

print("\n================ FINAL 5x5 CV WITH FIXED PARAMS ================")

final_results = evaluate_with_params(
    FIXED_PARAMS,
    record_fold_metrics=True,
)

print("\n================ FINAL SUMMARY ================")
print(f"Device used: {DEVICE_STR}")
print(f"Ordering time: {final_results['ordering_sec']:.2f} seconds")
print(f"Elapsed time for fixed-param 5x5 CV: {final_results['elapsed_sec']:.2f} seconds")
print(f"R2   : {final_results['r2_mean']:.6f} ± {final_results['r2_std']:.6f}")
print(f"RMSE : {final_results['rmse_mean']:.6f} ± {final_results['rmse_std']:.6f}")
print(f"MAE  : {final_results['mae_mean']:.6f} ± {final_results['mae_std']:.6f}")

final_results["fold_df"].to_csv("drivface_regression_fixed_5x5_foldwise.csv", index=False)

summary_df = pd.DataFrame([{
    "dataset": DATA_FILE,
    "target": TARGET_COL,
    "device": DEVICE_STR,
    "r2_mean": final_results["r2_mean"],
    "r2_std": final_results["r2_std"],
    "rmse_mean": final_results["rmse_mean"],
    "rmse_std": final_results["rmse_std"],
    "mae_mean": final_results["mae_mean"],
    "mae_std": final_results["mae_std"],
    "ordering_sec": final_results["ordering_sec"],
    "elapsed_sec": final_results["elapsed_sec"],
}])

summary_df.to_csv("drivface_regression_fixed_5x5_summary.csv", index=False)

print("\n[SAVED]")
print("  - drivface_regression_fixed_5x5_foldwise.csv")
print("  - drivface_regression_fixed_5x5_summary.csv")

[DATA] Raw shape=(606, 6401)
[DATA] Processed X=(606, 6400) | y=(606,)
[DATA] Numeric cols kept=6400 | Removed non-numeric cols=0
[GPU ] cuda_available= True | visible_gpus= 8 | using_device= cuda:2

================ FIXED PARAMETERS ================
go_metric: manhattan
go_num_clusters: 5
go_refine_passes: 1
go_direction_select: False
go_feat_subsample: 2000
nsc_segmentation: largest_jump
nsc_m_rule: idf
nsc_tau: 0.99
nsc_gamma: 2.654390393837633
nsc_beta: 0.043192175152615336
nsc_Mmin: 16
nsc_Mmax: 256
nsc_lmin: 12
assume_standardized: True
tabpfn_seed: 3

================ FINAL 5x5 CV WITH FIXED PARAMS ================
[Fold 01] R2=0.513978 | RMSE=10.775632 | MAE=4.802976
[Fold 02] R2=0.702793 | RMSE=6.697166 | MAE=2.874235
[Fold 03] R2=0.767134 | RMSE=7.360267 | MAE=3.627173
[Fold 04] R2=0.744861 | RMSE=7.592462 | MAE=3.400269
[Fold 05] R2=0.632208 | RMSE=7.979800 | MAE=3.195395
[Fold 06] R2=0.739993 | RMSE=7.371513 | MAE=3.586942
[Fold 07] R2=0.648237 | RMSE=9.118200 | MAE=4.29763